In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
df_bridges = pd.read_excel('../data/raw/BMMS_overview.xlsx')

In [ ]:
df_bridges.head(10)

In [ ]:
df_roads = pd.read_csv('../data/raw/_roads.tsv', sep='\t')

In [ ]:
df_roads.head(10)

In [ ]:
list(df_bridges.columns)

FIRST STEP OF IDENTIFYING BAD BRIDGES
- Check borders (they need to be inside Bangladesh)
- Check errors in file (some have location errors in the file )


In [ ]:
df_errors = df_bridges[df_bridges['EstimatedLoc'] == 'error']
df_errors.head(20)

In [ ]:
#rounded geographical borders from Bangledesch, with small buffer
LAT_MIN, LAT_MAX = 20.57 - 0.1, 26.63 + 0.1
LON_MIN, LON_MAX = 88.02 - 0.1, 92.68 + 0.1

#filter locations outside of Bangladesh
outside_bounds = (
    (df_bridges['lat'] < LAT_MIN) | (df_bridges['lat'] > LAT_MAX) |
    (df_bridges['lon'] < LON_MIN) | (df_bridges['lon'] > LON_MAX)
)

#filter error locations in file 
loc_error = df_bridges['EstimatedLoc'] == 'error'

#define the briders that are outside Bangladesh or have an error 
df_bridges['needs_repair'] = outside_bounds | loc_error

bad_bridges_count = df_bridges['needs_repair'].sum()
print(f"Number of bridges that need repair: {bad_bridges_count}")

df_bridges[df_bridges['needs_repair'] == True].head(10)

In [ ]:
df_lrp = pd.read_csv('../data/raw/Roads_InfoAboutEachLRP.csv')

#we use the bridges that are identified as needing repair for the next steps, so we create a new dataframe for that
df_to_repair = df_bridges[df_bridges['needs_repair'] == True].copy()

In [ ]:
def interpolate_location(row, lrp_data):
    #looks at the specified bridge we want to repair
    #remembers the road and the km point of the bridge
    road_id = row['road']
    bridge_km = row['km']
    
    # Filter LRP-data for specific road and sort by chainage 
    road_points = lrp_data[lrp_data['road'] == road_id].sort_values('chainage')
    
    # If there are no LRP points for this road, we cannot interpolate
    # we keep the old coordinates, but we also add a note in the EstimatedLoc column to indicate that this is an error that we could not fix
    if road_points.empty:
        return row['lat'], row['lon'], 'error_no_road_data'
    
    # Idenficate the two LRP-points around the bridge
    before = road_points[road_points['chainage'] <= bridge_km].tail(1)
    after = road_points[road_points['chainage'] >= bridge_km].head(1)
    
    #if the bridge is for example at km 0.5, but the first LRP point is at km 1, 
    #the funciton put the bridge to the closes known place at the road 
    if before.empty or after.empty:
        nearest = road_points.iloc[(road_points['chainage'] - bridge_km).abs().argsort()[:1]]
        return nearest['lat'].values[0], nearest['lon'].values[0], 'repaired_nearest_lrp'

    p1 = before.iloc[0]
    p2 = after.iloc[0]
    
    if p1['chainage'] == p2['chainage']:
        return p1['lat'], p1['lon'], 'repaired_exact_match'
    
    #the fraction is exactly in the middle of the two points, so we can do a simple linear interpolation to find the new coordinates for the bridge
    fraction = (bridge_km - p1['chainage']) / (p2['chainage'] - p1['chainage'])
    
    new_lat = p1['lat'] + fraction * (p2['lat'] - p1['lat'])
    new_lon = p1['lon'] + fraction * (p2['lon'] - p1['lon'])
    
    return new_lat, new_lon, 'repaired_interpolated'



In [ ]:
df_bridges_fixed = df_bridges.copy()

# Only use the function on bridges that need repair, the outliers were identified  in the previous steps, so we can focus on those for the interpolation
results = df_bridges_fixed[df_bridges_fixed['needs_repair'] == True].apply(
    lambda row: pd.Series(interpolate_location(row, df_lrp)), axis=1
)
df_bridges_fixed.loc[df_bridges_fixed['needs_repair'] == True, ['lat', 'lon', 'EstimatedLoc']] = results.values

In [ ]:
df_bridges_fixed[df_bridges_fixed['needs_repair'] == True].head(10)

In [ ]:
#check if the bridges are now within the bounds of Bangladesh
#this is not correct, but we need to use the new creaded LRP file, because this one still has LRP's outside the boundaries 

plt.figure(figsize=(8, 8))
plt.scatter(df_bridges_fixed['lon'], df_bridges_fixed['lat'], s=1, c='green', label='Repaired')
plt.scatter(df_bridges['lon'], df_bridges['lat'], s=1, c='red', alpha=0.3, label='Original')

plt.title("Location of Bridges in Bangladesh")
plt.legend()
plt.show()

In [ ]:
df_bridges['type'].value_counts()


In [ ]:
df_bridges[df_bridges['needs_repair'] == True]['type'].value_counts()